In [ ]:
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Load model and tokenizer
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, token=True)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype="auto")

# Define the chat function
def chat_fn(message, history):
    # Build prompt with full history
    prompt = "<|begin_of_text|>"
    for user_msg, assistant_msg in history:
        prompt += f"<|start_header_id|>user<|end_header_id|>\n{user_msg}<|eot_id|>\n"
        prompt += f"<|start_header_id|>assistant<|end_header_id|>\n{assistant_msg}<|eot_id|>\n"
    prompt += f"<|start_header_id|>user<|end_header_id|>\n{message}<|eot_id|>\n"
    prompt += f"<|start_header_id|>assistant<|end_header_id|>\n"

    # Generate response
    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
    response = pipe(prompt, max_new_tokens=512, do_sample=True, temperature=0.7)
    full_text = response[0]["generated_text"]

    # Extract just the assistant's reply (cut at <|eot_id|>)
    reply = full_text.split("<|eot_id|>")[0].split("<|start_header_id|>assistant<|end_header_id|>\n")[-1].strip()

    # Add to history and return
    history.append((message, reply))
    return "", history

c:\Users\Dell\miniconda3\envs\llama3env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model...


c:\Users\Dell\miniconda3\envs\llama3env\lib\site-packages\transformers\models\auto\tokenization_auto.py:898: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
c:\Users\Dell\miniconda3\envs\llama3env\lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell\.cache\huggingface\hub\models--meta-llama--Meta-Llama-3-8B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to

In [ ]:
# Gradio chat UI
gr.ChatInterface(
    fn=chat_fn,
    title="LLaMA 3 Chat",
    chatbot=gr.Chatbot(),
    textbox=gr.Textbox(placeholder="Ask LLaMA 3 something...", scale=7),
    theme="soft",
).launch()